In [3]:
# This allows us to import modules from the parent directory
import os
os.chdir("..")

import torch
from torch.amp import autocast
import time
import matplotlib.pyplot as plt
from model import ContinuousMotionModel
from dataset.dataset import *
import utils.utils as utils
import soundfile as sf
from IPython.display import clear_output
import os

device = utils.get_device()

In [13]:
# model_path = "trained_models/20-09-tests/rich_feature_speak_fingers_0__mv5ce2aw_epoch_2001.pth"
model_path = "trained_models/pretrained_sliding_diffusion.pth"

# Load the model
model: ContinuousMotionModel = ContinuousMotionModel.load_model(model_path, device)
model = model.to(device)
model.eval() # Set the model to evaluation mode

c:\python\311\Lib\site-packages\torch\nn\init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


RuntimeError: Error(s) in loading state_dict for ContinuousMotionModel:
	size mismatch for style_linear.weight: copying a param with shape torch.Size([64, 17]) from checkpoint, the shape in current model is torch.Size([64, 18]).

In [11]:
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
print(animation_visualisation.init_visualization(display=False))

http://localhost:8001/utils/animation/visualisation/new/animation_viewer.html?wsport=8702


In [ ]:
# Sliding Diffusion inference loop
with autocast(device_type=device.type, dtype=torch.bfloat16):
    dataset = GPUDataset(
        consolidated_file="dataset/genea2023_dataset/val/main-agent/consolidated.npz",
        seq_length=80,
        seed_length=8,
        batch_size=1,
        epoch_length=1,  # Set to 1 for testing purposes
        loading_encoded_data=False,
        include_world_pos_gesture_features=True,
        include_vel_acc_features=True,
        return_audio_frame_index=True,  # Set to True to return the audio frame index,
        include_speaking_status=False
    )

    # Use no gradient calculation for inference
    with torch.no_grad():

        gesture_sequence, seed_gesture, _, main_agent_id_one_hot, finger_availability, start_frames = [
            item.to(device) for item in next(iter(dataset))
        ]

        # main_agent_id_one_hot = torch.nn.functional.one_hot(torch.tensor([8]), num_classes=17).float().to(device)
        # finger_availability = torch.zeros_like(finger_availability).to(device)
        # print(finger_availability)

        full_audio_features = dataset.audio.to(device)
        start_frame = start_frames[0].item()  # The starting frame index in the consolidated data

        # Find the correct audio file based on the start_frame
        audio_file_path = None
        relative_start_frame = 0
        for i, segment in enumerate(dataset.metadata['file_segments']):
            file_start, file_end = segment['start_idx'], segment['end_idx']
            if file_start <= start_frame < file_end:
                # This is the correct file. Construct the path.
                # The prefix is constructed based on the file index.
                base_dir = os.path.dirname(dataset.consolidated_file)

                prefix = f"val_2023_v0_{i:03d}"

                wav_dir = os.path.join(base_dir, "wav")
                audio_file_name = f"{prefix}_main-agent.wav"
                audio_file_path = os.path.join(wav_dir, audio_file_name)
                relative_start_frame = start_frame - file_start
                break
        
        try:
            if audio_file_path:
                audio_data, samplerate = sf.read(audio_file_path)
                print(f"Successfully loaded audio from: {audio_file_path}")
            else:
                raise FileNotFoundError
        except (FileNotFoundError, TypeError):
            print(f"Could not find or load audio file for start frame {start_frame}. Audio will not be played.")
            audio_data = None

        # Decode the input using the autoencoder model
        if model.pose_encoder is not None:
            encoded_gesture_sequence = model.pose_encoder.encode(gesture_sequence)
        else:
            encoded_gesture_sequence = gesture_sequence

        iteration_counter = 0
        
        # --- Audio Playback Setup ---
        frame_duration = 1.0 / 30.0  # 30 FPS
        audio_delay_frames = -50  # Delay audio by 30 frames (1 second) to match gesture latency
        audio_chunk_duration = 0.5  # seconds
        overlap_duration = 0.1  # seconds
        last_chunk_time = -1.0  # Trigger first chunk immediately
        # --------------------------

        while True:
            iteration_counter += 1
            # Start time for the current frame
            frame_start_time = time.time()

            # The audio features also have to be shifted by one frame
            # I have the full audio features and the starting frame, so I extract the audio features for the current frame
            actual_audio_features = full_audio_features[start_frame + iteration_counter: start_frame + iteration_counter + dataset.seq_length, :].unsqueeze(0)

            # --- Stream Audio Chunk ---
            if audio_data is not None:
                # We use the relative_start_frame to calculate time within the specific audio file
                current_audio_time = (relative_start_frame + iteration_counter - audio_delay_frames) * frame_duration
                if current_audio_time >= last_chunk_time + (audio_chunk_duration - overlap_duration):
                    start_index = int((current_audio_time - overlap_duration) * samplerate)

                    start_index = max(0, start_index)
                    end_index = start_index + int(audio_chunk_duration * samplerate)
                    
                    if start_index < len(audio_data) and end_index <= len(audio_data):
                        audio_chunk = audio_data[start_index:end_index]
                        animation_visualisation.send_audio(audio_chunk, samplerate)
                        last_chunk_time = current_audio_time
            # --------------------------

            encoded_gesture_sequence, noisy_gesture_sequence = model.inference(encoded_gesture_sequence, actual_audio_features, main_agent_id_one_hot, finger_availability, seed_gesture)

            ########################################################################################################################################################################

            # Decode the output using the autoencoder model
            clear_output(wait=True)

            # In case the model added extra features for richer embeddings, we only take the original features for decoding
            denoised_frame_to_decode = encoded_gesture_sequence[:,model.diffusion.clean_frame_index,:model.original_pose_features_per_frame].unsqueeze(0)
            
            pre_decode_time = time.time()
            if model.pose_encoder is not None:
                unencoded_denoised_frame = model.pose_encoder.decode(denoised_frame_to_decode)
            else:
                unencoded_denoised_frame = denoised_frame_to_decode
            post_decode_time = time.time()
            
            print(f"Time taken for decoding in ms: {(post_decode_time - pre_decode_time) * 1000:.2f} ms")

            denmormalized_unencoded_denoised_frame = dataset.skeleton.denormalize_poses(unencoded_denoised_frame).squeeze(0).squeeze(0)
            animation_visualisation.send_pose(denmormalized_unencoded_denoised_frame.cpu(), dataset.skeleton)
            animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),noisy_gesture_sequence.squeeze(0)[:,:100].to(torch.float32)), dim=1), "full tensor")

            print(f"Iteration: {iteration_counter}")
            frame_end_time = time.time()

            frame_time = frame_end_time - frame_start_time

            # Print the time taken for the current frame
            print(f"Frame {iteration_counter} processed in {frame_time:.4f} seconds ({1/(frame_end_time - frame_start_time):.2f} FPS)")

            # Sleep for the remaining time in the 30 FPS frame
            time_to_sleep = max(0, (1/30) - frame_time - 0.0005)  # 0.01 is a small buffer to account for processing time
            time.sleep(time_to_sleep)

Time taken for decoding in ms: 0.00 ms


KeyboardInterrupt: 